# Sepsis population — **batched** generator (cardio-only, SI stack)

<details>
<summary>Generates a virtual-patient cohort by **Latin-Hypercube-sampling clinical targets per sepsis</summary>

phenotype** and calibrating the cardiovascular model to each sampled target set — one virtual
subject per sample (EFC paper §"Sepsis Population": 200 LHS × {normal, warmShock, coldShock}).

Mirrors `Convergence_Run_Batch.ipynb` (one vmapped solve per calibration stage on the SI stack,
streamed raw tensor, two-phase config→run→save→load→analyse) with **one structural change**: each
lane calibrates to its **own** target set. Since the batch injects per-lane values only into
**state** columns of `Y0`, the population model (`cvModel.json`) exposes the 16 controller
targets as **target-holding states** (`cubicStateController`), and the per-lane clinical targets
ride the existing `sampled_params` → `Y0` injection. The 16 controlled parameters start at the
base state and are solved by the controllers.

- Batching is the lever: 100s–1000s of subjects are one vmapped solve (fast even on CPU).
- Diverged lanes are expected (stiff calibration) and masked to `BAD_RUN_SENTINEL`.
- Toggle CPU/GPU and float64/float32 in the imports cell.

</details>

In [ ]:
# region -> runConfig: the single run-configuration surface
# The ONE place run configuration lives (project rule). Defined first so the
# device/precision block can be applied before JAX initialises below.
runConfig = {
    # --- file references ---
    "model":    "cvModel_linear.json",  # cardio-only model (16 cubicStateController + 16 target states)
    "scenario": "sepsis_linear.json",   # twin + calibration.stages + population ranges
    "mode":     "calibration",   # each member is a staged calibration run

    # --- pipeline phases ---
    "run":  True,  # Phase 1 — run the LHS sweep + save
    "plot": True,   # Phase 2 — load + analyse + plot

    # --- device / precision (applied in Imports, before `import jax`) ---
    "device": {
        "useGpu":    False,       # CPU here; True -> CUDA device
        "precision": "float64",   # "float64" (reference) or "float32"
    },

    # --- population sweep ---
    "population": {
        "perPhenotype": 400,      # LHS samples/phenotype; nrModels = perPhenotype * #phenotypes
        "seed":         0,        # LHS reproducibility
        "errorTarget":  0.5,      # SUCCESS if max |rel err| (%) <= this
    },

    # --- batched solve ---
    "solver":    {"type": "euler"}, # "rk4" or "euler"
    "chunkSize": 1200,               # samples per vmap + streaming granularity

    # --- calibration overrides ---
    "calibration": {},  # per-key overrides of the scenario calibration section; {} = as-is

    # --- analysis ---
    "analysis": {
        "atm":             760.0,   # atmospheric offset (gauge = raw - atm)
        "divergenceLimit": 2000.0,  # |value| >= this in any obs/param -> BAD run
    },

    # --- plotting ---
    "plotOpts": {
        "targetPoints": 1500,   # strided-decimation target for the full-raw fallback
    },

    # --- sim-time progress print ---
    "progressEvery": 10,   # print convergence line every N simulated seconds; 0 = off

    # --- raw-data saving ---
    "saveRaw":   True,   # stream every saved run's full trajectory to disk (large; async writer)

    # --- output ---
    "output": {
        "save": True, 
        "path": "notebookData/sepsis", 
        "name": "population_sepsis_flatPCs.h5",
        "logProgress": True # persist the live-progress trace into the .h5
    },   

    # post-processing / per-run HDF5 artifact off: the population file is the artifact
    "postProcessing": None,
    "requested":      None,
    "plots":          [],
    "printStatus":    True,
    "printEveryPct":  0,   # print chunk progress every this % of chunks

    # --- integration numerics (override scenario shared.integration) ---
    "runTime": 10,       # simulated seconds per internal run
    "dt":      0.0005,   # integrator step
    "dtDense": 1.0,      # save/output grid: 1.0 = 1 Hz; raise for within-beat waveforms
}
# endregion

## Imports

In [ ]:
# region -> imports + device/precision (must run before `import jax`)
# ---- repo-root bootstrap: run from any cwd (make `library` importable + resolve
# ---- the relative notebookData/ + config/ paths). Walks up to the dir containing library/. ----
import os, sys
_root = os.path.abspath(os.getcwd())
while not os.path.isdir(os.path.join(_root, "library")) and _root != os.path.dirname(_root):
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)
os.chdir(_root)

# ---- device / precision (from runConfig, MUST run before JAX initialises) -------
useGpu    = runConfig["device"]["useGpu"]
precision = runConfig["device"]["precision"]

import os
if useGpu:
    os.environ.pop("CUDA_VISIBLE_DEVICES", None)
    os.environ["JAX_PLATFORMS"] = "cuda"
    # GPU memory hygiene (must precede `import jax`): grow on demand instead of grabbing
    # ~75% of VRAM up front (so JAX coexists with the display on a small shared card), and
    # hand freed buffers back to the driver so the cleanup cell / del actually releases VRAM.
    os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
    os.environ["XLA_PYTHON_CLIENT_ALLOCATOR"]   = "platform"
else:
    os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
    os.environ["JAX_PLATFORMS"] = "cpu"

import jax
jax.config.update("jax_enable_x64", precision == "float64")

import numpy as np
from scipy.stats import qmc                      # Latin Hypercube sampling
import pandas as pd
import matplotlib.pyplot as plt
import json
import time

import library.run.runner as runner          # buildSimulationParams (+ calibratorUpdater)
import library.run.runnerBatchSI as runnerBatchSI   # batched (vmapped) SI calibration
import library.viz.plots as libPlots         # plotCalibrationConvergence
import library.utils as utils
from library.hdf5 import schema_pop                       # population artifact (init_population / final_states)
from library.hdf5.raw_stream import RawTraceStreamWriter  # async streaming writer for the full raw tensor
import library.postproc.reporting as reporting            # shared scope / rejection report

np.set_printoptions(suppress=True)
print("devices:", jax.devices(), "| x64:", jax.config.jax_enable_x64)
# endregion

## Assemble `simulationParams` + load the population config

<details>
<summary>`runner.buildSimulationParams` expands the slim `runConfig` + scenario into the legacy</summary>

`simulationParams` shape. The `population` block of the scenario carries the per-phenotype
clinical target ranges (LHS), the 16 controlled observations, their aligned **target-holding
states** (injected per lane), and any driver states (`Cyc_HC`). The 16 controlled *parameters*
are read from the calibration section — they are solved by the controllers, not swept.

</details>

In [ ]:
# region -> assemble simulationParams + load the population config
scenario = utils.loadScenario(runConfig["scenario"])
simulationParams = runner.buildSimulationParams(runConfig, scenario)

popCfg        = scenario["population"]
twin          = scenario["shared"]["twin"]["twinTargets"]
volDist       = scenario["shared"]["twin"]["volumeDistribution"]
phenotypes    = popCfg["phenotypes"]                 # {name: {clinicalDim: [lo, hi]}}
phenoNames    = list(phenotypes.keys())
clinicalOrder = popCfg["clinicalOrder"]              # the LHS-sampled clinical dims (order == columns)
observations  = popCfg["observations"]               # 16 controlled observations (each has a per-lane target)
targetStates  = popCfg["targetStates"]               # target-holding states, aligned to observations
driverStates  = popCfg["driverStates"]               # e.g. Cyc_HC (60/HR), injected per lane
controlledParams = scenario["calibration"]["adaptive"]["parameters"]   # 16 calibrated params (solved, not swept)

# The batch injects per-lane values only into STATE columns of Y0, so the swept "params" are the
# target states + driver states; the controlled parameters start at base and are calibrated.
param_names   = list(targetStates) + list(driverStates)

perPheno      = runConfig["population"]["perPhenotype"]
seed          = runConfig["population"]["seed"]
nrModels      = perPheno * len(phenoNames)
pop           = {"nrModels": nrModels, "seed": seed,
                 "errorTarget": runConfig["population"]["errorTarget"]}
outPath       = os.path.join(runConfig["output"]["path"], runConfig["output"]["name"])

print(f"phenotypes = {phenoNames} x {perPheno} = {nrModels} models | "
      f"injected states = {len(param_names)} | observations = {len(observations)}")
print(f"output -> {outPath}")
# endregion

## Clinical → model-target transform + atmospheric offsets

<details>
<summary>Each virtual subject is defined by a set of **clinical** targets (blood volume, systolic/pulse</summary>

pressures, CVP, HR, CO, capillary pressures). `clinicalToTargets` maps those to the 16 **model
target-state** values (the controller's compare space, gauge mmHg / mL / mL·min⁻¹) plus the
`Cyc_HC` driver (`60/HR`), following the EFC paper's conventions:

- `amp_P_As` is sampled as a **divisor** → pulse amplitude = `Sys_P_As / divisor`.
- `avg_P_Cs` is a **fractional pressure drop** from systemic diastolic toward CVP.
- `avg_P_Cp` is an **offset** below pulmonary diastolic.
- `SV = CO/HR`; compartment volumes = `volumeDistribution × TBV`.

Absolute-pressure observations carry the model's +760 mmHg offset, subtracted for gauge
comparison (`offsetArr`); amplitudes, volumes and SV carry no offset.

</details>

In [ ]:
# region -> clinical -> model-target transform + atmospheric offsets
ATM = runConfig["analysis"]["atm"]

def obsOffset(name):
    """Atmospheric offset baked into absolute-pressure signals (gauge = raw - offset)."""
    return utils.obsOffset(name, ATM)

# compartment for each volume observation's target state (avg_V_X -> volumeDistribution[X])
_VOL_COMP = {"avg_V_As": "As", "avg_V_Ap": "Ap", "avg_V_Hl": "Hl", "avg_V_Hr": "Hr",
             "avg_V_Cs": "Cs", "avg_V_Vt": "Vt", "avg_V_Cp": "Cp", "avg_V_Vp": "Vp"}

def clinicalToTargets(c):
    """One subject's 10 sampled clinical dims -> {target_<obs>: value} (gauge, the controller's
    compare space y[varTarget]-offset) + the Cyc_HC driver state. See the markdown above for the
    divisor / fractional-drop / offset conventions (EFC paper Table calibratorsSmall)."""
    amp   = c["Sys_P_As"] / c["amp_P_As"]              # pulse amplitude (divisor convention)
    diaAs = c["Sys_P_As"] - amp                        # systemic arterial diastolic
    t = {
        "target_avg_P_Vs":      c["CVP"],
        "target_avg_P_Cs":      diaAs - c["avg_P_Cs"] * (diaAs - c["CVP"]),   # fractional drop
        "target_avg_P_Cp":      c["Dia_P_Ap"] - c["avg_P_Cp"],               # offset below pulm dia
        "target_keep_max_P_As": c["Sys_P_As"],
        "target_keep_max_P_Ap": c["Sys_P_Ap"],
        "target_amp_P_As":      amp,
        "target_amp_P_Ap":      c["Sys_P_Ap"] - c["Dia_P_Ap"],
        "target_keep_SV_Hl":    c["CO"] / c["HR"],
    }
    for obs, comp in _VOL_COMP.items():
        t["target_" + obs] = c["TotalBloodVolume"] * volDist[comp]
    t["Cyc_HC"] = 60.0 / c["HR"]
    return t

offsetArr = np.array([obsOffset(o) for o in observations])
# sanity: every injected state has a transform entry (spot-check on the scenario's twin center)
_probe = clinicalToTargets({**{k: twin.get(k, 0.0) for k in clinicalOrder},
                            "amp_P_As": twin.get("amp_P_As", 2.5)})
missing = [p for p in param_names if p not in _probe]
assert not missing, f"transform missing target(s): {missing}"
print("transform OK for all", len(param_names), "injected states | offsets:",
      dict(zip(observations, offsetArr.astype(int))))
# endregion

## Per-phenotype Latin Hypercube sample of the clinical targets

<details>
<summary>For each phenotype, LHS-sample the clinical target ranges (`perPhenotype` subjects), transform to</summary>

the 16 model target states (+ `Cyc_HC`), and stack the cohorts. Each row becomes one lane's
injected `Y0` columns (`sampled_params`). `targetMatrix` keeps the per-lane gauge target for each
observation (used by the Phase-2 error summary); `phenotypeIdx` labels each lane's cohort. Each
phenotype uses a distinct LHS stream (`seed + cohort index`) for reproducibility.

</details>

In [ ]:
# region -> per-phenotype LHS sample -> injected states + per-lane targets
if runConfig["run"]:
    # Per-phenotype LHS over the clinical target ranges -> per-lane injected states + gauge targets.
    sampled_params = np.zeros((nrModels, len(param_names)))     # injected Y0 columns (target + driver states)
    targetMatrix   = np.zeros((nrModels, len(observations)))    # per-lane gauge target for each observation
    clinicalMat    = np.zeros((nrModels, len(clinicalOrder)))   # the raw sampled clinical dims (for inspection)
    phenotypeIdx   = np.zeros(nrModels, dtype=int)

    for pi, pname in enumerate(phenoNames):
        ranges  = phenotypes[pname]
        lo      = np.array([ranges[c][0] for c in clinicalOrder])
        hi      = np.array([ranges[c][1] for c in clinicalOrder])
        sampler = qmc.LatinHypercube(d=len(clinicalOrder), seed=seed + pi)   # distinct stream per cohort
        clin    = qmc.scale(sampler.random(perPheno), lo, hi)                # (perPheno, nClinical)
        for k in range(perPheno):
            row = pi * perPheno + k
            t   = clinicalToTargets(dict(zip(clinicalOrder, clin[k])))
            sampled_params[row] = [t[p] for p in param_names]
            targetMatrix[row]   = [t["target_" + o] for o in observations]
            clinicalMat[row]    = clin[k]
            phenotypeIdx[row]   = pi

    print("sampled_params:", sampled_params.shape, "| targetMatrix:", targetMatrix.shape,
          "| lanes/phenotype:", {n: int(np.sum(phenotypeIdx == i)) for i, n in enumerate(phenoNames)})
    pd.DataFrame(clinicalMat, columns=clinicalOrder).assign(
        phenotype=[phenoNames[i] for i in phenotypeIdx]).groupby("phenotype").head(2)
# endregion

## Run the population (single batched / vmapped solve + async raw streaming)

<details>
<summary>One `runnerBatchSI.batchedCalibration` call replaces the per-sample Python loop: the population</summary>

is integrated in **sample chunks** (`chunkSize`, which bounds VRAM), each chunk running the full
stage stack on device. Diverged lanes propagate non-finite values (no per-sample try/except) and
are masked to `BAD_RUN_SENTINEL`.

With `saveRaw=True`, **every saved run's full trajectory** (all states + algebraic outputs) is
streamed to the HDF5 `raw` dataset `(N, T_total, C)` by a background writer thread — the
compressed disk write overlaps the next chunk's GPU compute, so RAM/VRAM stay bounded by
`chunkSize` while the full (tens-of-GB) raw tensor lands on disk. The file is created up front
(`init_population`) so the writer can append `raw` immediately.

</details>

In [ ]:
# region -> run the batched population + write artifact/timings
if runConfig["run"]:
    if runConfig["output"]["save"]:
        os.makedirs(runConfig["output"]["path"], exist_ok=True)

    # Converged-window signals for the plots: observations + the calibrated (controlled) parameters.
    traceNames = list(dict.fromkeys(list(observations) + list(controlledParams)))
    # Realized per-column sampling extent of the injected states (target states + Cyc_HC) -> problem.
    bounds = np.stack([sampled_params.min(axis=0), sampled_params.max(axis=0)], axis=1)  # (P, 2)

    # --- set up the population file up front (init_population must precede the raw writer) ---
    # One scalar prep is shared by the file setup and the run.
    prep = runnerBatchSI.prepare(simulationParams)
    layout = runnerBatchSI.rawLayout(simulationParams, prepared=prep)
    stateNames = layout["stateNames"]
    rawDtype = "float64" if jax.config.jax_enable_x64 else "float32"

    saveRaw = runConfig.get("saveRaw", False) and runConfig["output"]["save"]
    if runConfig["output"]["save"]:
        schema_pop.init_population(
            outPath, param_names=param_names, state_names=stateNames,
            observation_names=observations, sampled_params=sampled_params,
            model_structure=utils.modelStructureJSON(prep["modelStructure"]),
            problem={"names": param_names, "bounds": bounds.tolist(), "num_vars": len(param_names)},
            conf=runConfig,
            meta={"twinTargets": twin, "phenotypes": phenoNames, "perPhenotype": perPheno,
                  "phenotypeIdx": phenotypeIdx.tolist(), "clinicalOrder": clinicalOrder,
                  "phenotypeRanges": phenotypes, "targetStates": list(targetStates),
                  "driverStates": list(driverStates), "controlledParams": list(controlledParams)})

    def rawWriterFactory(signalNames, totalPoints, time_, nDense):
        print(f"  raw writer: ({pop['nrModels']}, {totalPoints}, {len(signalNames)}) {rawDtype} "
              f"+ gzip  (~{pop['nrModels'] * totalPoints * len(signalNames) * (8 if rawDtype=='float64' else 4) / 1e9:.1f} GB uncompressed)")
        return RawTraceStreamWriter(outPath, N=pop["nrModels"], signalNames=signalNames,
                                    totalPoints=totalPoints, nDense=nDense, time=time_, dtype=rawDtype)

    t0 = time.time()
    batch = runnerBatchSI.batchedCalibration(
        simulationParams, sampled_params, param_names, observations,
        chunkSize=runConfig.get("chunkSize", 64), printStatus=runConfig.get("printStatus", True),
        printEveryPct=runConfig.get("printEveryPct"), traceNames=traceNames,
        rawWriterFactory=(rawWriterFactory if saveRaw else None), prepared=prep)
    totalWall = time.time() - t0
    print(f"Batched population complete in {totalWall:.1f}s "
          f"(N={pop['nrModels']}, solver={simulationParams['solver']['type']}, "
          f"x64={jax.config.jax_enable_x64}, saveRaw={saveRaw}) -> {outPath}")

    finalStates = batch["finalStates"]                 # (N, nState)
    modelStructure = batch["modelStructure"]
    traces, traceT = batch["traces"], batch["traceT"]  # {name: (N, nT)}, (nT,)

    # Gauge observation matrix (raw final value minus atmospheric offset == serial steadyState).
    obsMatrix = batch["rawObs"] - offsetArr            # (N, nObs)

    # Out-of-scope mask: a lane whose observation OR calibrated parameter went non-finite or left
    # scope is a BAD run. Set those rows to NaN so the error-summary good-mask (below) drops them
    # exactly as the serial path does. The calibrated params ride the final-state columns.
    sIdxFS      = {n: i for i, n in enumerate(stateNames)}
    paramMatrix = np.column_stack([finalStates[:, sIdxFS[p]] for p in controlledParams if p in sIdxFS])
    finiteRow = schema_pop.good_run_mask(obsMatrix, runConfig["analysis"].get("divergenceLimit", 1e6),
                                         param_matrix=paramMatrix)
    obsMatrix[~finiteRow] = np.nan
    print(f"in-scope lanes: {finiteRow.sum()}/{pop['nrModels']}")

    # --- final_states table (NN/Table-5); full traces already streamed to top-level `raw` -----
    # write_final_states writes the table in one shot WITHOUT creating runs/{id} groups (the raw
    # tensor is the per-run store, so empty runs/{id}/raw groups would only mislead).
    if runConfig["output"]["save"]:
        fsArr = finalStates.copy()
        fsArr[~finiteRow] = schema_pop.BAD_RUN_SENTINEL
        schema_pop.write_final_states(outPath, fsArr, [str(i) for i in range(pop["nrModels"])])
        print(f"final_states written -> {outPath}  | raw tensor "
              f"{(pop['nrModels'], layout['totalSavedPoints'], len(layout['signalNames']))}"
              f"  (read via schema_pop.raw_trace / f['raw'])")
        # Batched wall clock: one vmapped solve, so total wall (+ amortized total/N) is the
        # meaningful cost -- no per-run distribution. Stored as a 1-element timings array.
        schema_pop.write_timings(outPath, [totalWall], meta={
            "device":     "gpu" if useGpu else "cpu",
            "precision":  precision,
            "solver":     simulationParams["solver"]["type"],
            "dt":         simulationParams["dt"],
            "runTime":    simulationParams["runTime"],
            "nrModels":   pop["nrModels"],
            "stack":      "SI",
            "total_wall": totalWall,
        })
        # persist the live-progress trace (the convergence lines the solve printed) for comparison
        schema_pop.write_progress(outPath, batch.get("progress"), meta={
            "model": runConfig["model"], "scenario": runConfig["scenario"],
            "mode": runConfig["mode"], "solver": simulationParams["solver"]["type"],
            "nrModels": pop["nrModels"], "stack": "SI"})
# endregion

# Phase 2 — Load & analyse (from the saved file)

<details>
<summary>Everything below reconstructs from the population `.h5` — no in-session run state</summary>

is required. After a kernel restart you can run the setup cells (config →
`simulationParams` → targets → LHS, all scenario-only and cheap) then the **load**
cell below and every analysis/plot/table cell, without re-running the sweep.

The load cell rebuilds `modelStructure` (from the stored `model_structure` JSON) and
`obsMatrix` / `finiteRow` (each observation's converged value = last saved `raw`
timepoint minus its atmospheric offset), plus the batch timings (`timingMeta`)
written in Phase 1. The full per-run trajectories stay on disk in the `raw` tensor,
which the convergence plots read directly.

</details>

In [ ]:
# region -> Phase 2 -- load everything from the saved file
if runConfig["plot"]:
    # --- load everything the analysis below needs, straight from the saved file ----------
    # Reconstructs the in-memory run state (modelStructure, obsMatrix, per-lane targetMatrix,
    # phenotype labels, finiteRow) so Phase 2 is independent of Phase 1. The per-lane gauge targets
    # ARE the injected target-state columns of sampled_params, so nothing extra needs storing.
    import h5py

    with h5py.File(outPath, "r") as f:
        modelStructure = json.loads(f["model_structure"].asstr()[()])
        sig            = list(f["raw_signal_names"].asstr()[:])
        observations   = list(f["observation_names"].asstr()[:])
        pnF            = list(f["param_names"].asstr()[:])
        sampled_params = np.asarray(f["sampled_params"][:])
        meta           = json.loads(f.attrs["meta"]) if "meta" in f.attrs else {}
        src            = "raw_coarse" if "raw_coarse" in f else "raw"
        lastRow        = np.asarray(f[src][:, -1, :])     # (N, C) converged (run-end) values

    sigIdx = {n: i for i, n in enumerate(sig)}
    Nrun   = lastRow.shape[0]
    obsMatrix = np.full((Nrun, len(observations)), np.nan)
    for j, o in enumerate(observations):
        if o in sigIdx:
            obsMatrix[:, j] = lastRow[:, sigIdx[o]] - obsOffset(o)

    # per-lane targets = the injected target-state columns; phenotype labels + controlled params from meta.
    pnIdx        = {p: i for i, p in enumerate(pnF)}
    targetMatrix = np.column_stack([sampled_params[:, pnIdx["target_" + o]] for o in observations])
    phenotypeIdx = np.asarray(meta.get("phenotypeIdx", np.zeros(Nrun, int)))
    phenoNames   = meta.get("phenotypes", ["all"])
    controlledParams = meta.get("controlledParams", [])

    # run-end calibrated-parameter values -> checked for out-of-scope alongside the observations
    # (a run whose observation stays finite but whose solved param blows up is still a BAD run).
    paramInScope = [p for p in controlledParams if p in sigIdx]
    paramMatrix  = (np.column_stack([lastRow[:, sigIdx[p]] for p in paramInScope])
                    if paramInScope else np.zeros((Nrun, 0)))

    finiteRow = schema_pop.good_run_mask(obsMatrix, runConfig["analysis"].get("divergenceLimit", 1e6),
                                         param_matrix=paramMatrix)
    obsMatrix[~finiteRow] = np.nan
    traces, traceT, saveRaw = None, None, True            # plots read the file (raw/raw_coarse) below

    runWall, timingMeta = schema_pop.read_timings(outPath)
    print(f"loaded {Nrun} runs from {outPath} | in-scope lanes {finiteRow.sum()}/{Nrun} "
          f"| phenotypes {phenoNames} | timings: {'yes' if timingMeta else 'none'}")
# endregion

## Scope / rejection report

<details>
<summary>Breaks down which runs were dropped and **why**, before the error summary works on the survivors.</summary>

A run is out of scope when **any** observation *or* calibrated parameter reaches `|value| >=
divergenceLimit` (`runConfig.analysis.divergenceLimit`), or the lane is NaN/Inf or sentinel-stamped.
Reports the total kept/dropped split, the per-reason counts (sentinel / NaN / observation-out-of-scope
/ parameter-out-of-scope — a run can trip more than one), a per-phenotype kept/dropped table, and the
individual signals that drove the out-of-scope drops with their worst run-end magnitude.

</details>

In [ ]:
# region -> scope / rejection report (why each run was dropped)
if runConfig["plot"]:
    lim    = runConfig["analysis"].get("divergenceLimit", 1e6)
    # rawObs rebuilt un-blanked from the run-end tensor slice (obsMatrix is blanked in load).
    rawObs = np.column_stack([
        (lastRow[:, sigIdx[o]] - obsOffset(o)) if o in sigIdx else np.full(Nrun, np.nan)
        for o in observations])
    rep = reporting.scopeRejectionReport(rawObs, paramMatrix, observations, paramInScope, lim=lim)
# endregion

In [ ]:
# region -> per-phenotype kept / dropped breakdown (sepsis-specific)
if runConfig["plot"]:
    # extends the shared scope report with the sepsis cohort split, keyed off rep.keep.
    dropped  = ~rep.keep
    perPheno = pd.DataFrame({
        "phenotype": phenoNames,
        "total":   [int(np.sum(phenotypeIdx == pi)) for pi in range(len(phenoNames))],
        "kept":    [int(np.sum(rep.keep & (phenotypeIdx == pi))) for pi in range(len(phenoNames))],
        "dropped": [int(np.sum(dropped   & (phenotypeIdx == pi))) for pi in range(len(phenoNames))],
    })
    print("per-phenotype kept / dropped:")
    print(perPheno.to_string(index=False))
# endregion

## Error summary

<details>
<summary>Filter faulty runs (any observation == `BAD_RUN_SENTINEL`), then compute each subject's</summary>

**per-lane** relative error `(obs - target) / target × 100` against its own sampled target
(`targetMatrix`). A subject is SUCCESS when `max |relative error| <= errorTarget`. Convergence
counts are also broken out **per phenotype**.

</details>

In [ ]:
# region -> error summary (per-lane, per-phenotype)
if runConfig["plot"]:
    good = schema_pop.good_run_mask(obsMatrix, runConfig["analysis"].get("divergenceLimit", 1e6),
                                    param_matrix=paramMatrix)
    print(f"{good.sum()}/{len(good)} valid runs ({(~good).sum()} faulty/out-of-scope dropped)")

    obsGood     = obsMatrix[good]
    tgtGood     = targetMatrix[good]                       # per-lane targets (aligned to observations)
    phenoGood   = phenotypeIdx[good]
    errRel      = (obsGood - tgtGood) / tgtGood * 100.0    # per-lane relative error
    errAbs      = obsGood - tgtGood
    obsTargeted = list(observations)                       # every observation has a per-lane target

    success = np.max(np.abs(errRel), axis=1) <= pop["errorTarget"]
    print(f"converged (max|rel err| <= {pop['errorTarget']}%): {success.sum()}/{len(success)}")
    for pi, pn in enumerate(phenoNames):                   # per-phenotype valid + success counts
        tot = int(np.sum(phenotypeIdx == pi))
        m   = phenoGood == pi
        print(f"  {pn:10s}: valid {int(m.sum())}/{tot} | success {int(np.sum(success[m]))}/{int(m.sum())}")

    summary = pd.DataFrame({
        "observation": obsTargeted,
        "mean_target": np.nanmean(tgtGood, axis=0),
        "mean_rel_err_%": np.nanmean(errRel, axis=0),
        "std_rel_err_%": np.nanstd(errRel, axis=0),
        "mean_abs_err": np.nanmean(errAbs, axis=0),
    })
    summary
# endregion

## Plots

In [ ]:
# region -> boxplot of per-observation relative error
if runConfig["plot"]:
    # --- boxplot: per-lane relative error distribution per observation ------------------
    if good.sum() > 0:
        fig, ax = plt.subplots(figsize=(12, 5))
        ax.boxplot(errRel, tick_labels=utils.labelsFor(obsTargeted, "latex"), showfliers=False)
        ax.axhline(0.0, color="k", lw=0.8)
        ax.axhline(pop["errorTarget"], color="r", ls="--", lw=0.8, label=f"±{pop['errorTarget']}% target")
        ax.axhline(-pop["errorTarget"], color="r", ls="--", lw=0.8)
        ax.set_ylabel("relative error (%)")
        ax.set_title(f"Sepsis population calibration error across {good.sum()} valid subjects "
                     f"({', '.join(phenoNames)})")
        ax.tick_params(axis="x", rotation=90)
        ax.legend()
        plt.tight_layout()
        plt.show()
# endregion

## LaTeX summary table

<details>
<summary>Builds the Table-5-style summary (each targeted observation paired with the parameter that</summary>

controls it), rendered with paper-quality names from `config/labels.json` via
`utils.labelFor`, and written to `notebookData/convergence_summary.tex`.

</details>

In [ ]:
# region -> LaTeX summary table
if runConfig["plot"]:
    # --- Table-5-style LaTeX summary -----------------------------------------------------
    # Each targeted observation paired with the calibration parameter that controls it
    # (controller varTarget -> controlled param), with paper-quality names from config/labels.json.
    calib = modelStructure["calibration"]
    paramForObs = {calib[p]["params"]["varTarget"]: p
                   for p in controlledParams if p in calib}     # observation -> controlling param

    arrFS, stateNames = schema_pop.final_states_array(outPath)   # (N, S) calibrated final states
    sIdx = {n: i for i, n in enumerate(stateNames)}
    paramVals = {p: arrFS[good, sIdx[p]] for p in controlledParams if p in sIdx}

    meanTarget = np.nanmean(targetMatrix[good], axis=0)          # cohort-mean per-lane target
    rows = {}
    for j, o in enumerate(obsTargeted):
        p = paramForObs.get(o, "")
        pv = paramVals.get(p)
        has_p = pv is not None and pv.size > 0
        rows[utils.labelFor(o, "latex")] = [
            f"{meanTarget[j]:.4g}",
            f"{summary['mean_rel_err_%'][j]:.3g}",
            f"{summary['std_rel_err_%'][j]:.2g}",
            utils.labelFor(p, "latex") if p else "--",
            f"{np.mean(pv):.4g}" if has_p else "--",
            f"{np.std(pv):.2g}" if has_p else "--",
        ]

    latexTable = utils.generate_latex_table_new(
        rows,
        ["Vars", "Mean target", "Rel. Error \\%", "std", "Param", "Value", "std"],
        "", "", "sepsisPopulation",
        f"Summary of the sepsis population across {good.sum()} calibrated virtual subjects "
        f"({', '.join(phenoNames)}). For each calibration target the table reports the cohort-mean "
        f"per-subject target, the mean signed relative error and its standard deviation at "
        f"convergence, together with the corresponding calibrated parameter values.")

    out_tex = os.path.join(runConfig["output"]["path"], "sepsis_population_summary.tex")
    with open(out_tex, "w") as f:
        f.write(latexTable)
    print(latexTable)
    print(f"\nLaTeX table written to: {out_tex}")
# endregion

## Observation convergence plot

<details>
<summary>One panel per **targeted observation**, overlaying each subject's trajectory over the **whole</summary>

calibration**. Unlike the single-twin convergence study, every subject has its **own** target, so
no single target line is drawn — the panels show the cohort of trajectories settling (per-subject
error is in the boxplot / summary table above).

Reads the tiny **`raw_coarse`** companion (one converged value per saved run) — instant. If a file
has only the full `raw` tensor, it falls back to a single strided pass; build the companion once
with `schema_pop.build_raw_coarse(outPath)`. Only converged subjects are drawn.

</details>

In [ ]:
# region -> observation convergence plot
if runConfig["plot"]:
    if good.sum() > 0:
        # --- per-subject observation trajectory over the WHOLE calibration (no single target line) --
        # Prefer the tiny `raw_coarse` companion (one converged value per saved run) -> instant.
        # Else fall back to a single strided pass over the full `raw` tensor (slow: decompresses it all).
        import h5py

        calib = modelStructure["calibration"]
        paramForObs  = {calib[p]["params"]["varTarget"]: p for p in controlledParams if p in calib}
        offsetsByObs = {o: off for o, off in zip(observations, offsetArr)}
        goodIdx = np.where(finiteRow)[0]
        targetPoints = runConfig["plotOpts"]["targetPoints"] # strided-decimation target for the full-raw fallback

        goodTraces, plotT, titleScope = {}, traceT, "converged window"
        if saveRaw and runConfig["output"]["save"]:
            with h5py.File(outPath, "r") as f:
                if "raw_coarse" in f:                                   # fast path
                    src, tkey, ds, titleScope = "raw_coarse", "raw_coarse_time", 1, "whole calibration"
                else:                                                   # slow fallback (full raw)
                    src, tkey = "raw", "raw_time"
                    ds, titleScope = max(1, f["raw"].shape[1] // targetPoints), "whole calibration (full raw)"
                sig = list(f["raw_signal_names"].asstr()[:])
                names = [o for o in obsTargeted if o in sig]
                block = f[src][goodIdx, ::ds, :]                        # good runs only, one pass
                plotT = f[tkey][::ds]
            goodTraces = {o: block[:, :, sig.index(o)] for o in names}

        # per-lane targets differ across the cohort -> no `targets=` overlay; offsets still apply.
        libPlots.plotCalibrationConvergence(
            goodTraces or {o: traces[o][finiteRow] for o in obsTargeted if o in (traces or {})},
            traceT=plotT, offsets=offsetsByObs, paramForObs=paramForObs,
            title=f"Sepsis population convergence ({titleScope}, {simulationParams['solver']['type']})")
        plt.show()
# endregion

## Parameter convergence plot

<details>
<summary>Companion to the observation plot, for the 16 **calibrated (controlled) parameters** — the free</summary>

parameters the controllers solve for. One panel per parameter, overlaying each converged subject's
parameter trajectory (jet). No LHS-edge lines (these parameters are *calibrated*, not sampled) and
no legend; the spread across the cohort shows the phenotype-driven parameter heterogeneity the
paper reports (e.g. systemic resistance / compliance shifts between warm and cold shock).

</details>

In [ ]:
# region -> parameter convergence plot
if runConfig["plot"]:
    if good.sum() > 0:
        # --- one panel per CALIBRATED parameter over the whole calibration -------------------------
        # Calibrated params have no LHS edges (they are solved, not sampled) -> no `ranges=`.
        # Same source order as the observation plot: raw_coarse (instant) -> full raw (strided).
        import h5py

        goodIdx = np.where(finiteRow)[0]
        targetPoints = runConfig["plotOpts"]["targetPoints"] # strided-decimation target for the full-raw fallback

        paramTraces, plotT, titleScope = {}, traceT, "converged window"
        if saveRaw and runConfig["output"]["save"]:
            with h5py.File(outPath, "r") as f:
                if "raw_coarse" in f:                                   # fast path
                    src, tkey, ds, titleScope = "raw_coarse", "raw_coarse_time", 1, "whole calibration"
                else:                                                   # slow fallback (full raw)
                    src, tkey = "raw", "raw_time"
                    ds, titleScope = max(1, f["raw"].shape[1] // targetPoints), "whole calibration (full raw)"
                sig = list(f["raw_signal_names"].asstr()[:])
                names = [p for p in controlledParams if p in sig]
                block = f[src][goodIdx, ::ds, :]                        # good runs only, one pass
                plotT = f[tkey][::ds]
            paramTraces = {p: block[:, :, sig.index(p)] for p in names}
        else:
            paramTraces = {p: traces[p][finiteRow] for p in controlledParams if p in (traces or {})}

        libPlots.plotCalibrationConvergence(
            paramTraces, traceT=plotT, paramForObs=None, showLegend=False,
            title=f"Sepsis population -- calibrated parameters ({titleScope}, {simulationParams['solver']['type']})")
        plt.show()
# endregion

## Run timings

<details>
<summary>Wall-clock cost of the batched sweep, loaded from the population file (`timingMeta`).</summary>

Because the whole population is one vmapped solve, the meaningful numbers are the
**total wall** for `nrModels` calibrations and the **amortized** seconds-per-calibration
(total ÷ N) — there is no per-run distribution to plot. Directly supports the reviewer's
timing ask; cross-file CPU-vs-GPU and scaling-vs-N comparisons live in the dedicated
timing-analysis notebook.

</details>

In [ ]:
# region -> batched timing summary
if runConfig["plot"]:
    # --- batched timing summary (total wall + amortized per-calibration) -----------------
    if timingMeta:
        totalWall = timingMeta.get("total_wall")
        n = timingMeta.get("nrModels") or pop["nrModels"]
        timingSummary = pd.DataFrame([{
            "device":          timingMeta.get("device", "?"),
            "precision":       timingMeta.get("precision", "?"),
            "solver":          timingMeta.get("solver", "?"),
            "dt":              timingMeta.get("dt"),
            "runTime":         timingMeta.get("runTime"),
            "nrModels":        n,
            "total_wall_s":    totalWall,
            "amortized_s/run": (totalWall / n) if (totalWall and n) else None,
        }])
        display(timingSummary)
    else:
        print("no timings in file (Phase 1 ran with output.save=False)")
# endregion

## (Deferred) NN training set

<details>
<summary>The population file is the input to the inverse-problem calibration NN (X = achieved observations</summary>

`obsMatrix` or the intended `targetMatrix`, y = calibrated parameters from `final_states`).
Deferred this round; the hook is:

```python
from library.hdf5 import schema_pop, schema_calib
arr, state_names = schema_pop.final_states_array(outPath)
training = schema_calib.build_training_set(
    arr, state_names, observation_keys=observations, param_keys=controlledParams)
schema_calib.save_calibration_run("notebookData/calibration_sepsis.h5", **training)
```

Note: this needs the observation values stored as columns in `final_states`; the current run
stores observations only in the `raw` tensor. Wire the steady-state observation values into the
`final_state` row (or a parallel table) before enabling this.

</details>

In [ ]:
# region -> release memory / cleanup
# ---- release GPU memory --------------------------------------------------------------
# Drop references to the big result arrays, clear JAX's compiled caches, and (with
# XLA_PYTHON_CLIENT_ALLOCATOR=platform from the device cell) hand the VRAM back to the
# driver -- no kernel restart needed. A kernel restart is still the guaranteed full reset.
import gc
for _v in ("batch", "results", "finalStates", "traces", "traceT", "obsMatrix", "targetMatrix",
           "prep", "layout", "sampled_params", "clinicalMat", "finiteRow", "fsArr",
           "goodTraces", "paramTraces", "block", "runWall", "lastRow"):
    globals().pop(_v, None)
jax.clear_caches()
gc.collect()
try:
    used = sum((d.memory_stats() or {}).get("bytes_in_use", 0) for d in jax.devices())
    print(f"GPU bytes in use after cleanup: {used/1e6:.0f} MB (restart kernel for a full reset)")
except Exception as e:
    print("memory_stats() unavailable on this device:", e)
# endregion